In [1]:
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent.parent
sys.path.insert(0, str(BASE))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from src.data import Data
import src.metrics as metrics

d:\venv\time-series-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
## Get the train data.
# Check if it already exists
if not os.path.exists(r"../../../data/single-daywise-sales-list-train.pkl"):
    print("Premade training data not found, cooking it...")
    data = Data()

    prod_id_list = data.get_prod_id_list()
    NUM_DATA = 100
    
    d_train = pd.DataFrame({"Id": prod_id_list.iloc[0:NUM_DATA],
                        "Day_Wise_Sales": [np.array(data.get_train_data(prod, choice1='train', choice2='single_prod_day_wise'))[0] for prod in tqdm(prod_id_list.iloc[0:NUM_DATA])]                   
                        })
    # Since loading data is slow, store it in pkl
    print("Saving training data for quicker future use")
    d_train.to_pickle("../../../data/single-daywise-sales-list-train.pkl")
    print("Training Data saved successfully")
else:
    print("Premade training data found!, loading it...")
    d_train = pd.read_pickle("../../../data/single-daywise-sales-list-train.pkl")
    print("Training Data loaded successfully")


## Similarly get the val data.
if not os.path.exists(r"../../../data/single-daywise-sales-list-val.pkl"):
    print("Premade validation data not found, cooking it...")
    data = Data()

    prod_id_list = data.get_prod_id_list()
    NUM_DATA = 100
    
    d_val = pd.DataFrame({"Id": prod_id_list.iloc[0:NUM_DATA],
                        "Day_Wise_Sales": [np.array(data.get_train_data(prod, choice1='validation', choice2='single_prod_day_wise'))[0] for prod in tqdm(prod_id_list.iloc[0:NUM_DATA])]                   
                        })
    # Since loading data is slow, store it in pkl
    print("Saving validation data for quicker future use")
    d_val.to_pickle("../../../data/single-daywise-sales-list-val.pkl")
    print("Validation Data saved successfully")
else:
    print("Premade validation data found!, loading it...")
    d_val = pd.read_pickle("../../../data/single-daywise-sales-list-val.pkl")
    print("Validation Data loaded successfully")


Premade training data found!, loading it...
Training Data loaded successfully
Premade validation data found!, loading it...
Validation Data loaded successfully


In [5]:
d_train['Min'] = d_train['Day_Wise_Sales'].apply(np.min)
d_train['Max'] = d_train['Day_Wise_Sales'].apply(np.max)
d_train['Normalized_Day_Wise_Sales'] = (d_train['Day_Wise_Sales'] - d_train['Min'])/ (d_train['Max']  - d_train['Min'])

d_val['Min'] = d_val['Day_Wise_Sales'].apply(np.min)
d_val['Max'] = d_val['Day_Wise_Sales'].apply(np.max)
d_val['Normalized_Day_Wise_Sales'] = (d_val['Day_Wise_Sales'] - d_val['Min'])/ (d_val['Max']  - d_val['Min'])

In [6]:
def create_sequences(arr, window_size):
    x_sequences = []
    y_sequences = []
    for i in range(len(arr) - window_size):
        x_sequences.append(arr[i : i + window_size])
        y_sequences.append(arr[i + window_size])
    return x_sequences, y_sequences

In [7]:
create_sequences(d_train['Normalized_Day_Wise_Sales'].iloc[0], 10)

([array([0. , 0. , 0. , 0. , 0. , 0.2, 0. , 0. , 0. , 0. ]),
  array([0. , 0. , 0. , 0. , 0.2, 0. , 0. , 0. , 0. , 0. ]),
  array([0. , 0. , 0. , 0.2, 0. , 0. , 0. , 0. , 0. , 0. ]),
  array([0. , 0. , 0.2, 0. , 0. , 0. , 0. , 0. , 0. , 0. ]),
  array([0. , 0.2, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]),
  array([0.2, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]),
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
  array([0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.4]),
  array([0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.4, 0. ]),
  array([0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.4, 0. , 0.2]),
  array([0. , 0. , 0. , 0. , 0. , 0. , 0.4, 0. , 0.2, 0.2]),
  array([0. , 0. , 0. , 0. , 0. , 0.4, 0. , 0.2, 0.2, 0. ]),
  array([0. , 0. , 0. , 0. , 0.4, 0. , 0.2, 0.2, 0. , 0. ]),
  array([0. ,

In [14]:
class GRU_Model(nn.Module):
    def __init__(self, input_dim = 1, hidden_dim = 64, num_layers = 1):
        super().__init__()

        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first= True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x) # out -> (batch_size, time, hidden_dim)
        return self.fc(out[:, -1, :])


In [9]:
def train_model(model, train_data_loader, val_data_loader, optimizer, loss_criterion, epochs=10, device=device):
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0
        for X_batch, y_batch in train_data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)

            loss = loss_criterion(y_batch, outputs)
            loss.backward()

            optimizer.step()

            running_train_loss += (loss.item() * X_batch.size(0))
        epoch_train_loss = running_train_loss / len(train_data_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_data_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                outputs = model(X_batch)

                loss = loss_criterion(y_batch, outputs)

                running_val_loss += (loss.item() * X_batch.size(0))
        epoch_val_loss = running_val_loss / len(val_data_loader.dataset)
        val_losses.append(epoch_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}]  "
            f"Train Loss: {epoch_train_loss:.4f}  "
            f"Val Loss: {epoch_val_loss:.4f}")

    return train_losses, val_losses

In [10]:
def forecast(model, x_train, horizon, device):
    with torch.no_grad():
        x_train = x_train.to(device)
        model.eval()
        y_pred = []
        for _ in range(horizon):
            output = model(x_train)
            y_pred.append(output.item())

            x_train = torch.cat([x_train[:, 1:, :], output.unsqueeze(-1)], dim = 1)
        return y_pred

In [15]:
model = GRU_Model(input_dim=1, hidden_dim=64, num_layers=1)
optimizer = optim.Adam(model.parameters(), lr = 1e-4)
loss = nn.MSELoss()

In [17]:
BATCH_SIZE = 64
NUM_DATAPOINTS = 1
WINDOW_SIZE = 1
HORIZON = 28
metric_names = [
    'MAE',
    'RMSE',
    'WAPE',
    'WRMSE',
    'mean_error'
    ]

avg_scores = {m : [] for m in metric_names}

for ts_train, ts_val in zip(d_train['Normalized_Day_Wise_Sales'][:NUM_DATAPOINTS], d_val['Normalized_Day_Wise_Sales'][:NUM_DATAPOINTS]):
    X_train, y_train = create_sequences(ts_train, WINDOW_SIZE)
    X_val, y_val = create_sequences(ts_val, WINDOW_SIZE)

    train_dataloader = DataLoader(TensorDataset(torch.tensor(X_train).to(torch.float32).unsqueeze(-1), torch.tensor(y_train).to(torch.float32).unsqueeze(-1)), batch_size= BATCH_SIZE)
    val_dataloader = DataLoader(TensorDataset(torch.tensor(X_val).to(torch.float32).unsqueeze(-1), torch.tensor(y_val).to(torch.float32).unsqueeze(-1)), batch_size= BATCH_SIZE)

    train_losses, val_losses = train_model(model, train_dataloader, val_dataloader, optimizer, loss)
    
    y_pred = np.array(forecast(model, torch.tensor(X_train[-1]).to(torch.float32).unsqueeze(-1).unsqueeze(0), HORIZON, device))
    y_true = np.array(y_val[len(y_train) : len(y_train) + HORIZON])
    for met in metric_names:
        avg_scores[met].append(getattr(metrics, met)(y_pred, y_true))
    

results_df = pd.DataFrame({
    "Metric": metric_names,
    "Average Score": [
        np.mean(avg_scores[met])
        for met in metric_names
    ]
})

print(results_df.to_string(index=False))

Epoch [1/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [2/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [3/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [4/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [5/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [6/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [7/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [8/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [9/10]  Train Loss: 0.0252  Val Loss: 0.0288
Epoch [10/10]  Train Loss: 0.0252  Val Loss: 0.0288
    Metric  Average Score
       MAE       0.172474
      RMSE       0.250608
      WAPE       1.527771
     WRMSE       1.326091
mean_error       0.051393
